In [1]:
#import pacakages
import requests
import os
import pandas as pd
import pyodbc
from sqlalchemy import create_engine
from datetime import datetime
import urllib
import schedule
import logging
import time

In [2]:
# -----------------------------
# CONFIGURATION
# -----------------------------
# Prefer reading from env var but fall back to the known key for convenience
FRED_API_KEY = os.getenv("FRED_API_KEY", "d175e91c5228f6cf634d4d1d85536be6")
FRED_SERIES_ID = "CUSR0000SAS4"  # Example: US CPI Transportation
SQL_SERVER = "LAPTOP-89MS9SPM"       # e.g., "MYPC\\SQLEXPRESS"
SQL_DATABASE = "ETLPracticeData"
SQL_TABLE = "CPITransportationData"

In [3]:
# -----------------------------
# EXTRACT
# -----------------------------
def extract_fred_data(series_id: str, api_key: str) -> pd.DataFrame:
    """Fetch data from FRED API and return as DataFrame."""
    if not api_key:
        raise ValueError("FRED API key is missing. Set FRED_API_KEY environment variable.")

    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id": series_id,
        "api_key": api_key,
        "file_type": "json"
    }

    try:
        response = requests.get(url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
    except requests.RequestException as e:
        raise RuntimeError(f"Error fetching data from FRED API: {e}")

    if "observations" not in data:
        raise ValueError("Unexpected API response format.")

    return pd.DataFrame(data["observations"])

In [4]:
# -----------------------------
# TRANSFORM
# -----------------------------
def transform_data(df: pd.DataFrame) -> pd.DataFrame:
    """Clean and prepare data for loading."""
    if df.empty:
        raise ValueError("No data to transform.")

    df = df[["date", "value"]].copy()
    df.rename(columns={"value": "cpit"}, inplace=True)
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["cpit"] = pd.to_numeric(df["cpit"], errors="coerce")
    df.dropna(subset=["date", "cpit"], inplace=True)

    return df

In [5]:
# -----------------------------
# LOAD
# -----------------------------
def load_to_sqlserver(df: pd.DataFrame, server: str, database: str, table: str):
    """Load DataFrame into SQL Server using Windows Authentication."""
    if df.empty:
        raise ValueError("No data to load.")

    conn_str = (
        f"DRIVER={{ODBC Driver 18 for SQL Server}};"
        f"SERVER={server};"
        f"DATABASE={database};"
        "Trusted_Connection=yes;"
    )

    try:
        conn = pyodbc.connect(conn_str)
        cursor = conn.cursor()

        # Create table if not exists
        cursor.execute(f"""
            IF NOT EXISTS (SELECT * FROM sysobjects WHERE name='{table}' AND xtype='U')
            CREATE TABLE {table} (
                date DATE PRIMARY KEY,
                cpit FLOAT
            )
        """)
        conn.commit()

        # Upsert data
        for _, row in df.iterrows():
            cursor.execute(
                f"MERGE {table} AS target "
                f"USING (SELECT ? AS date, ? AS cpit) AS source "
                f"ON target.date = source.date "
                f"WHEN MATCHED THEN UPDATE SET cpit = source.cpit "
                f"WHEN NOT MATCHED THEN INSERT (date, cpit) VALUES (source.date, source.cpit);",
                row["date"], row["cpit"]
            )

        conn.commit()
        print(f"[{datetime.now()}] ✅ Loaded {len(df)} rows into {table}.")

    except pyodbc.Error as e:
        raise RuntimeError(f"Error loading data into SQL Server: {e}")
    finally:
        if 'conn' in locals():
            conn.close()

In [6]:
# -----------------------------
# ETL JOB
# -----------------------------
def run_etl():
    """Run the full ETL process."""
    try:
        print(f"[{datetime.now()}] 📥 Starting ETL job...")
        raw_df = extract_fred_data(FRED_SERIES_ID, FRED_API_KEY)
        clean_df = transform_data(raw_df)
        load_to_sqlserver(clean_df, SQL_SERVER, SQL_DATABASE, SQL_TABLE)
        print(f"[{datetime.now()}] 🎯 ETL job completed successfully.\n")
    except Exception as e:
        print(f"[{datetime.now()}] ❌ ETL job failed: {e}\n")



In [ ]:
# -----------------------------
# SCHEDULER
# -----------------------------
if __name__ == "__main__":
    # Schedule ETL to run every day at 08:00 AM
    schedule.every().day.at("08:00").do(run_etl)

    print("📅 ETL Scheduler started. Waiting for next run...")
    while True:
        schedule.run_pending()
        time.sleep(30)  # Check every 30 seconds

📅 ETL Scheduler started. Waiting for next run...
[2026-02-12 14:17:15.477499] 📥 Starting ETL job...
[2026-02-12 14:17:17.308828] ✅ Loaded 839 rows into CPITransportationData.
[2026-02-12 14:17:17.317358] 🎯 ETL job completed successfully.

